## Dependencies

In [2]:
# Install dependencies
%pip install langchain langchain-core langchain-openai langchain-community langchain-text-splitters chromadb pypdf python-dotenv -q

# Verify installation
import sys
print(f"Python version: {sys.version}")

# Verify key packages are installed
try:
    import langchain
    import langchain_core
    print(f"✅ LangChain version: {langchain.__version__}")
    print("✅ All core packages installed successfully!")
except ImportError as e:
    print(f"⚠️  Error: {e}")
    print("Please ensure all packages are installed correctly.")


Note: you may need to restart the kernel to use updated packages.
Python version: 3.13.12 (v3.13.12:1cbe4818347, Feb  3 2026, 13:36:53) [Clang 16.0.0 (clang-1600.0.26.6)]
✅ LangChain version: 1.2.12
✅ All core packages installed successfully!


## Setup & Imports

In [ ]:
# Standard library imports
import os
from pathlib import Path
from dotenv import load_dotenv

# LangChain core imports
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate

# Modern LangChain agent imports (latest pattern from LangChain docs)
from langchain.agents import create_agent
from langchain.tools import tool

# Load environment variables - check current directory first, then parent
current_dir = Path.cwd()
env_path = current_dir / '.env'
if env_path.exists():
    load_dotenv(env_path)
    print(f"✅ Loading .env from: {env_path}")
else:
    # Try parent directory
    parent_env = current_dir.parent / '.env'
    if parent_env.exists():
        load_dotenv(parent_env)
        print(f"✅ Loading .env from: {parent_env}")
    else:
        # Fallback to default behavior (searches current and parent directories)
        load_dotenv()
        print("⚠️  No .env file found. Using default load_dotenv() search")

# Verify API key is set
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    print("⚠️  WARNING: OPENAI_API_KEY not found in environment variables!")
    print("Please create a .env file in this directory with your OpenAI API key:")
    print(f"   {current_dir / '.env'}")
    print("\nFormat: OPENAI_API_KEY=sk-...")
else:
    # Validate API key format (basic check)
    if not api_key.startswith('sk-'):
        print("⚠️  WARNING: API key format looks incorrect. Should start with 'sk-'")
    else:
        print("✅ OpenAI API key loaded successfully!")
        # Test the API key by making a simple call
        try:
            test_llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0, max_tokens=5)
            test_llm.invoke("Hi")
            print("✅ API key validated successfully!")
        except Exception as e:
            print(f"❌ ERROR: API key validation failed!")
            print(f"   Error: {str(e)}")
            print("\n💡 Please check:")
            print("   1. Your API key is correct (get it from https://platform.openai.com/api-keys)")
            print("   2. You have sufficient credits in your OpenAI account")
            print("   3. The .env file is in the correct location")
            raise

print("\n✅ All imports successful!")
print("📚 Using modern LangChain patterns: create_agent with @tool decorator")

## Step 1: Load

- Use a LangChain document loader appropriate for your data type
- Print: number of documents loaded, sample of first document content

## Step 2: Chunk

- Use RecursiveCharacterTextSplitter
- Print: total chunks created, character count of smallest and largest chunk
- Experiment: Try at least 2 different chunk_size values (e.g., 500 vs 1000) and note what changes

## Step 3: Embed + Store

- Embed chunks using an embedding model of your choice
- Store in ChromaDB (or another vector DB if you're feeling adventurous)

# Step 4: Test Retrieval (before wiring up the LLM!)

- Run 3 test queries using similarity_search
- For each query, print the top 3 retrieved chunks
- Annotate: Are the retrieved chunks actually relevant? Yes/No and why?

## Step 5: Build the RAG Chain

- Wire up RetrievalQA with a custom prompt template
- Run the same 3 queries through the full chain
- Print the generated answers

## Step 6: Evaluate (the part most people skip)

- Create a mini eval set: 5 questions where you know the correct answer
- Run all 5 through your pipeline
- For each, record:

    ✅ or ❌ Did it retrieve the right chunks?
    ✅ or ❌ Is the answer grounded in the context (not hallucinated)?
    ✅ or ❌ Is the answer actually correct?
- Calculate your accuracy: X/5 retrieval, X/5 faithfulness, X/5 correctness